# Convert between transverse mercator and WGS84

The python code was rewritten from the JavaScript [proj4js library](https://github.com/proj4js/proj4js).

## The code

In [1]:
import math

def pj_enfn(es_t):
    C00 = 1             # 1
    C02 = 1 / 4         # 0.25
    C04 = 3 / 64        # 0.046875
    C06 = 5 / 256       # 0.01953125
    C08 = 7 / 65536     # 0.01068115234375
    C22 = 3 / 4         # 0.75
    C44 = 15 / 32       # 0.46875
    C46 = 5 / 384       # 0.01302083...
    C48 = 175 / 24576   # 0.00712076822916...
    C66 = 35 / 96       # 0.364583...
    C68 = 35 / 6144     # 0.005696614583...
    C88 = 315 / 1024    # 0.3076171875
    en = list((0, 0, 0, 0, 0))
    en[0] = C00 - es_t * (C02 + es_t * (C04 + es_t * (C06 + es_t * C08)))
    en[1] = es_t * (C22 - es_t * (C04 + es_t * (C06 + es_t * C08)))
    en[2] = es_t * es_t * (C44 - es_t * (C46 + es_t * C48))
    en[3] = es_t * es_t * es_t * (C66 - es_t * C68)
    en[4] = es_t * es_t * es_t * es_t * C88
    return en

def pj_mlfn(phi_t, en_t):
    cphi = math.cos(phi_t)
    sphi = math.sin(phi_t)
    cphi *= sphi
    sphi *= sphi
    return (en_t[0] * phi_t - cphi * (en_t[1] + sphi * (en_t[2] + sphi * (en_t[3] + sphi * en_t[4]))))

def pj_inv_mlfn(arg, es_t, en_t):
    k = 1 / (1 - es_t)
    phi_t = arg
    for i in range(20):
        s = math.sin(phi_t)
        t = 1 - es_t * s * s
        t = (pj_mlfn(phi_t, en_t) - arg) * (t * math.sqrt(t)) * k
        phi_t-= t
        if (abs(t) < 1E-10):
            return phi_t
    return phi_t

def adjust_lon(x):
    return x if (abs(x) <= 3.14159265359) else (x - (x / abs(x) * math.pi * 2))

def geodeticToGeocentric(lon, lat, a_t, es_t):
    if (lat < - math.pi / 2) & (lat > -1.001 * math.pi / 2):
        lat = - math.pi / 2

    elif (lat > math.pi / 2) & (lat < 1.001 * math.pi / 2):
        lat = math.pi / 2

    elif (lat < - math.pi / 2):
        return - math.inf, - math.inf, 0
    
    elif (lat > math.pi / 2):
        return math.inf, math.inf, 0
    
    if lon > math.pi:
        lon -= 2 * math.pi

    clat = math.cos(lat)
    slat = math.sin(lat)
    s2lat = slat * slat
    Rn = a_t / math.sqrt(1 - es_t * s2lat)
    x_t = Rn * clat * math.cos(lon)
    y_t = Rn * clat * math.sin(lon)
    z_t = Rn * (1 - es_t) * slat
    return x_t, y_t, z_t

def geocentricToGeodetic(x, y, z, a_t, es_t):
    genau = 1E-12
    genau2 = genau * genau
    P = math.sqrt(x * x + y * y)
    RR = math.sqrt(x * x + y * y + z * z)

    if (P / a_t < genau):
        lon = 0
        if (RR / a_t < genau):
            return x, y
    else:
        lon = math.atan2(y, x)

    CT = z / RR
    ST = P / RR
    RX = 1 / math.sqrt(1 - es_t * (2 - es_t) * ST * ST)
    CPHI0 = ST * (1 - es_t) * RX
    SPHI0 = CT * RX

    for i in range(30):
        RN = a_t / math.sqrt(1 - es_t * SPHI0 * SPHI0)
        height = P * CPHI0 + z * SPHI0 - RN * (1 - es_t * SPHI0 * SPHI0)
        RK = es_t * RN / (RN + height)
        RX = 1 / math.sqrt(1 - RK * (2 - RK) * ST * ST)
        CPHI = ST * (1 - RK) * RX
        SPHI = CT * RX
        SDPHI = SPHI * CPHI0 - CPHI * SPHI0
        CPHI0 = CPHI
        SPHI0 = SPHI
        if (SDPHI * SDPHI <= genau2):
            break
    
    lat = math.atan(SPHI / abs(CPHI))
    return lon, lat

def geocentricFromWgs84(x, y, z, param):
    if len(param) == 3:
        return x - param[0], y - param[1], z - param[2]
    
    elif len(param) == 7:
        Dx_BF = param[0]
        Dy_BF = param[1]
        Dz_BF = param[2]
        Rx_BF = param[3] * math.pi / 180 / 3600
        Ry_BF = param[4] * math.pi / 180 / 3600
        Rz_BF = param[5] * math.pi / 180 / 3600
        M_BF  = param[6] / 1E6 + 1
        x_tmp = (x - Dx_BF) / M_BF
        y_tmp = (y - Dy_BF) / M_BF
        z_tmp = (z - Dz_BF) / M_BF
        x_t = x_tmp + Rz_BF * y_tmp - Ry_BF * z_tmp
        y_t = - Rz_BF * x_tmp + y_tmp + Rx_BF * z_tmp
        z_t = Ry_BF * x_tmp - Rx_BF * y_tmp + z_tmp
        return x_t, y_t, z_t
    
    else:
        return x, y, z

def geocentricToWgs84(x, y, z, param):
    if len(param) == 3:
        return x + param[0], y + param[1], z + param[2]
    
    elif len(param) == 7:
        Dx_BF = param[0]
        Dy_BF = param[1]
        Dz_BF = param[2]
        Rx_BF = param[3] * math.pi / 180 / 3600
        Ry_BF = param[4] * math.pi / 180 / 3600
        Rz_BF = param[5] * math.pi / 180 / 3600
        M_BF  = param[6] / 1E6 + 1
        x_t = M_BF * (x - Rz_BF * y + Ry_BF * z) + Dx_BF
        y_t = M_BF * (Rz_BF * x + y - Rx_BF * z) + Dy_BF
        z_t = M_BF * (- Ry_BF * x + Rx_BF * y + z) + Dz_BF
        return x_t, y_t, z_t
    
    else:
        return x, y, z

def forward(lon, lat, x0, y0, xf, yf, sf, a, es, trans = None):
    if (trans):
        x_t, y_t, z_t = geodeticToGeocentric(lon / 180 * math.pi, lat / 180 * math.pi, 6378137, 0.00669437999014131699613723354004)
        x_t1, y_t1, z_t1 = geocentricFromWgs84(x_t, y_t, z_t, trans)
        lon_t, lat_t = geocentricToGeodetic(x_t1, y_t1, z_t1, a, es)
        lon = lon_t * 180 / math.pi
        lat = lat_t * 180 / math.pi

    dLon = adjust_lon((lon - x0) / 180 * math.pi)
    en = pj_enfn(es)
    cos_phi = math.cos(lat / 180 * math.pi)
    sin_phi = math.sin(lat / 180 * math.pi)

    al = cos_phi * dLon
    als = al * al
    c = es * cos_phi * cos_phi
    cs = c * c
    tq = math.tan(lat / 180 * math.pi) if abs(cos_phi) > 1E-10 else 0
    t = tq * tq
    ts = t * t
    con = 1 - es * sin_phi * sin_phi
    al = al / math.sqrt(con)
    ml = pj_mlfn(lat / 180 * math.pi, en)
    ml0 = pj_mlfn(y0 / 180 * math.pi, en)

    x = a * (sf * al * (1 +
      als / 6 * (1 - t + c +
      als / 20 * (5 - 18 * t + ts + 14 * c - 58 * t * c +
      als / 42 * (61 + 179 * ts - ts * t - 479 * t))))) + (
      xf)
    
    y = a * (sf * (ml - ml0 +
      sin_phi * dLon * al / 2 * (1 +
      als / 12 * (5 - t + 9 * c + 4 * cs +
      als / 30 * (61 + ts - 58 * t + 270 * c - 330 * t * c +
      als / 56 * (1385 + 543 * ts - ts * t - 3111 * t)))))) + (
      yf)
    
    return x, y

def inverse(x, y, x0, y0, xf, yf, sf, a, es, trans = None):
    con: float
    en = pj_enfn(es)
    lat: float
    lon: float
    ml0 = pj_mlfn(y0 / 180 * math.pi, en)
    phi: float
    X = (x - xf) / a
    Y = (y - yf) / a

    con = ml0 + Y / sf
    phi = pj_inv_mlfn(con, es, en)

    if (abs(phi) < (math.pi / 2)):
        cos_phi = math.cos(phi)
        sin_phi = math.sin(phi)
        tan_phi = math.tan(phi) if abs(cos_phi) > 1E-10 else 0
        c = es * cos_phi * cos_phi
        cs = c * c
        t = tan_phi * tan_phi
        ts = t * t
        con = 1 - es * sin_phi * sin_phi
        d = X * math.sqrt(con) / sf
        ds = d * d
        con *= tan_phi

        lat = phi - (con * ds / (1 - es)) * 0.5 * (1 -
            ds / 12 * (5 + 3 * t - 9 * c * t + c - 4 * cs -
            ds / 30 * (61 + 90 * t - 252 * c * t + 45 * ts + 46 * c -
            ds / 56 * (1385 + 3633 * t + 4095 * ts + 1574 * ts * t))))
        
        lon = adjust_lon(x0 / 180 * math.pi + (d * (1 -
            ds / 6 * (1 + 2 * t + c -
            ds / 20 * (5 + 28 * t + 24 * ts + 8 * c * t + 6 * c -
            ds / 42 * (61 + 662 * t + 1320 * ts + 720 * ts * t)))) / cos_phi))

        if (trans):
            x_t, y_t, z_t = geodeticToGeocentric(lon, lat, a, es)
            x_t, y_t, z_t = geocentricToWgs84(x_t, y_t, z_t, trans)
            lon, lat = geocentricToGeodetic(x_t, y_t, z_t, 6378137, 0.00669437999014131699613723354004)

        lat = round(lat * 180 / math.pi, 8)
        lon = round(lon * 180 / math.pi, 8)
        return lon, lat
    
def show_result(tmc, wgsc, tmo, tmf, tmsf, a, es, name, trans = None):
    f = forward(wgsc[0], wgsc[1], tmo[0], tmo[1], tmf[0], tmf[1], tmsf, a, es, trans)
    i = inverse(tmc[0], tmc[1], tmo[0], tmo[1], tmf[0], tmf[1], tmsf, a, es, trans)
    print(name)
    print('---')
    print(f'WGS84({wgsc[0]}, {wgsc[1]}) -> TM({f[0]}, {f[1]}) [{tmc[0] - f[0]}, {tmc[1] - f[1]}]')
    print(f'TM({tmc[0]}, {tmc[1]}) -> WGS84({i[0]}, {i[1]}) [{wgsc[0] - i[0]}, {wgsc[1] - i[1]}]\n')

## Ellipsoid definition

In [2]:
# Clarke 1866 ellipsoid
ellip_clrk66 = [6378206.4, 6356583.7999999994, 294.978698213898]
ellip_clrk66.append((2 - 1 / ellip_clrk66[2]) / ellip_clrk66[2])    # Eccentricity

# Everest 1830 ellipsoid
ellip_evrst30 = [6377276.345, 6356075.4131402399, 300.8017]
ellip_evrst30.append((2 - 1 / ellip_evrst30[2]) / ellip_evrst30[2]) # Eccentricity

# GRS80 ellipsoid
ellip_grs80  = [6378137, 6356752.3141403558, 298.257222101]
ellip_grs80.append((2 - 1 / ellip_grs80[2]) / ellip_grs80[2])       # Eccentricity

# International 1924 ellipsoid
ellip_intl24 = [6378388, 6356911.946127946, 297]
ellip_intl24.append((2 - 1 / ellip_intl24[2]) / ellip_intl24[2])    # Eccentricity

# WGS84 ellipsoid
ellip_wgs84 = [6378137, 6356752.314245, 298.257223563]
ellip_wgs84.append((2 - 1 / ellip_wgs84[2]) / ellip_wgs84[2])       # Eccentricity

## Performance in several countries

### East Asia

Country / Region included:

* China
* Hong Kong
* Japan
* Korea
* Taiwan

In [3]:
# China - CGCS2000 / 3-degree Gauss-Kruger zone 40 (EPSG: 4528)
tm_coord = (40595506.55628892, 3431403.6100768936)
wgs_coord = (121, 31)
tm_origin = (120, 0)
tm_false = (40500000, 0)
tm_sf = 1
show_result(tm_coord, wgs_coord, tm_origin, tm_false, tm_sf, ellip_grs80[0], ellip_grs80[3], 'CGCS2000 / 3-degree Gauss-Kruger zone 40 [EPSG: 4528]')

# Hong Kong - HK1980 Grid (EPSG: 2326)
tm_coord = (848955.3190933523, 817900.9157186308)
wgs_coord = (114.3, 22.3)
tm_origin = (114.178555555556, 22.3121333333333)
tm_false = (836694.05, 819069.8)
tm_sf = 1
tm_trans = [-162.619, -276.959, -161.764, 0.067753, -2.24365, -1.15883, -1.09425]
show_result(tm_coord, wgs_coord, tm_origin, tm_false, tm_sf, ellip_intl24[0], ellip_intl24[3], 'HK1980 Grid [EPSG: 2326]', tm_trans)

# Japan - JGD2011 / Plane Rectangular IX (EPSG: 6677)
tm_coord = (15025.785184915549, 12.845549761177372)
wgs_coord = (140, 36)
tm_origin = (139.833333333333, 36)
tm_false = (0, 0)
tm_sf = 0.9999
show_result(tm_coord, wgs_coord, tm_origin, tm_false, tm_sf, ellip_grs80[0], ellip_grs80[3], 'JGD2011 / Plane Rectangular IX [EPSG: 6677]')

# Korea - KGD2002 / Central Belt 2010 (EPSG: 5186)
tm_coord = (200000, 489012.95569100516)
wgs_coord = (127, 37)
tm_origin = (127, 38)
tm_false = (200000, 600000)
tm_sf = 1
show_result(tm_coord, wgs_coord, tm_origin, tm_false, tm_sf, ellip_grs80[0], ellip_grs80[3], 'KGD2002 / Central Belt 2010 [EPSG: 5186]')

# Taiwan - TWD97 / TM2 zone 121 (EPSG: 3826)
tm_coord = (351745.0803991604, 2655384.2884810874)
wgs_coord = (122, 24)
tm_origin = (121, 0)
tm_false = (250000, 0)
tm_sf = 0.9999
show_result(tm_coord, wgs_coord, tm_origin, tm_false, tm_sf, ellip_grs80[0], ellip_grs80[3], 'TWD97 / TM2 zone 121 [EPSG: 3826]')

CGCS2000 / 3-degree Gauss-Kruger zone 40 [EPSG: 4528]
---
WGS84(121, 31) -> TM(40595506.55617084, 3431403.610088034) [0.00011807680130004883, -1.1140480637550354e-05]
TM(40595506.55628892, 3431403.6100768936) -> WGS84(121.0, 31.0) [0.0, 0.0]

HK1980 Grid [EPSG: 2326]
---
WGS84(114.3, 22.3) -> TM(848955.3191290302, 817900.915744027) [-3.567792009562254e-05, -2.5396235287189484e-05]
TM(848955.3190933523, 817900.9157186308) -> WGS84(114.3, 22.3) [0.0, 0.0]

JGD2011 / Plane Rectangular IX [EPSG: 6677]
---
WGS84(140, 36) -> TM(15025.785184506249, 12.845549759271213) [4.092999006388709e-07, 1.9061587863689056e-09]
TM(15025.785184915549, 12.845549761177372) -> WGS84(140.0, 36.0) [0.0, 0.0]

KGD2002 / Central Belt 2010 [EPSG: 5186]
---
WGS84(127, 37) -> TM(200000.0, 489012.9556892108) [0.0, 1.7943675629794598e-06]
TM(200000, 489012.95569100516) -> WGS84(127.0, 37.0) [0.0, 0.0]

TWD97 / TM2 zone 121 [EPSG: 3826]
---
WGS84(122, 24) -> TM(351745.08023684064, 2655384.288484894) [0.0001623197458684

### South East Asia

Country included:

* Indonesia
* Malaysia
* Philippines
* Singapore
* Thailand
* Vietnam

In [4]:
# Indonesia - SRGI2013 / UTM zone 48S (EPSG: 9488)
tm_coord = (721383.3697866765, 9336391.399849912)
wgs_coord = (107, -6)
tm_origin = (105, 0)
tm_false = (500000, 10000000)
tm_sf = 0.9996
show_result(tm_coord, wgs_coord, tm_origin, tm_false, tm_sf, ellip_wgs84[0], ellip_wgs84[3], 'SRGI2013 / UTM zone 48S [EPSG: 9488]')

# Malaysia - GDM2000 / Selangor Grid (EPSG: 3380)
# Not Transverse Mercator! It's Cassini projection.
tm_coord = (-78092.48940223787, -19235.42956356777)
wgs_coord = (101, 3)
tm_origin = (101.389107913889, 3.68464905)
tm_false = (-34836.161, 56464.049)
tm_sf = 1
show_result(tm_coord, wgs_coord, tm_origin, tm_false, tm_sf, ellip_wgs84[0], ellip_wgs84[3], 'GDM2000 / Selangor Grid [EPSG: 3380]')

# Philippines - PRS92 / Philippines zone 3 (EPSG: 3123)
tm_coord = (499854.82345805597, 1658974.1646437685)
wgs_coord = (121, 15)
tm_origin = (121, 0)
tm_false = (500000, 0)
tm_sf = 0.99995
tm_trans = [-127.62, -67.24, -47.04, -3.068, 4.903, 1.578, -1.06]
show_result(tm_coord, wgs_coord, tm_origin, tm_false, tm_sf, ellip_clrk66[0], ellip_clrk66[3], 'PRS92 / Philippines zone 3 [EPSG: 3123]', tm_trans)

# Singapore - SVY21 / Singapore TM (EPSG: 3414)
tm_coord = (46550.1732962637, 31373.525578659588)
wgs_coord = (104, 1.3)
tm_origin = (103.833333333333, 1.36666666666667)
tm_false = (28001.642, 38744.572)
tm_sf = 1
show_result(tm_coord, wgs_coord, tm_origin, tm_false, tm_sf, ellip_wgs84[0], ellip_wgs84[3], 'SVY21 / Singapore TM [EPSG: 3414]')

# Thailand - Indian 1975 / UTM zone 47N (EPSG: 24047)
tm_coord = (717436.6829558045, 1437634.808684584)
wgs_coord = (101, 13)
tm_origin = (99, 0)
tm_false = (500000, 0)
tm_sf = 0.9996
tm_trans = [293, 836, 318, 0.5, 1.6, -2.8, 2.1]
show_result(tm_coord, wgs_coord, tm_origin, tm_false, tm_sf, ellip_evrst30[0], ellip_evrst30[3], 'Indian 1975 / UTM zone 47N [EPSG: 24047]', tm_trans)

# Vietnam - VN-2000 / TM-3 106-00 (EPSG: 9211)
tm_coord = (499804.3558246642, 2322953.9106769813)
wgs_coord = (106, 21)
tm_origin = (106, 0)
tm_false = (500000, 0)
tm_sf = 0.9999
tm_trans = [-191.90441429, -39.30318279, -111.45032835, 0.00928836, -0.01975479, 0.00427372, 0.252906278]
show_result(tm_coord, wgs_coord, tm_origin, tm_false, tm_sf, ellip_wgs84[0], ellip_wgs84[3], 'VN-2000 / TM-3 106-00 [EPSG: 9211]', tm_trans)

SRGI2013 / UTM zone 48S [EPSG: 9488]
---
WGS84(107, -6) -> TM(721383.1435225438, 9336391.42468776) [0.22626413265243173, -0.02483784779906273]
TM(721383.3697866765, 9336391.399849912) -> WGS84(107.00000204, -6.00000022) [-2.0399999982601003e-06, 2.2000000043931323e-07]

GDM2000 / Selangor Grid [EPSG: 3380]
---
WGS84(101, 3) -> TM(-78092.82321031686, -19235.429565472747) [0.3338080789835658, 1.9049766706302762e-06]
TM(-78092.48940223787, -19235.42956356777) -> WGS84(101.000003, 3.0) [-3.000000006636583e-06, 0.0]

PRS92 / Philippines zone 3 [EPSG: 3123]
---
WGS84(121, 15) -> TM(499854.8234580562, 1658974.16464544) [-2.3283064365386963e-10, -1.67149119079113e-06]
TM(499854.82345805597, 1658974.1646437685) -> WGS84(120.99999999, 15.00000001) [9.999993721976352e-09, -1.000000082740371e-08]

SVY21 / Singapore TM [EPSG: 3414]
---
WGS84(104, 1.3) -> TM(46550.173295083616, 31373.52557865927) [1.1800875654444098e-06, 3.1650415621697903e-10]
TM(46550.1732962637, 31373.525578659588) -> WGS84(104.0